# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their @id values. 

Please note: In Croissant, record sets, fields, and columns are referenced by their `@id` attribute, which ensures explicit and unambiguous identification.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Total record sets found: {len(record_sets)}\n")

for recset in record_sets:
    print(f"RecordSet @id: {recset.id}")
    print(f"  Name: {recset.name}")
    print(f"  Description: {getattr(recset, 'description', 'No description')}\n  Fields:")
    for field in recset.fields:
        print(f"    - Field @id: {field.id}, Column @id: {field.column.id if hasattr(field, 'column') else 'N/A'}  (name: {getattr(field, 'name', None)})")
    print('-'*50)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

We will extract data from all available record sets and organize each as a DataFrame.

In [ ]:
# Extract data from each record set into DataFrames, referenced by @id
dataframes = {}
record_set_ids = [recset.id for recset in dataset.record_sets]
print(f"Record sets loaded: {record_set_ids}\n")

for recset_id in record_set_ids:
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"RecordSet @id: {recset_id}")
    print("  Columns:", df.columns.tolist())
    print("  Sample:")
    display(df.head(3))


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

Please refer to each record set's schema to use field `@id` values for reference in analysis. Below is an example EDA workflow using one record set. Modify as appropriate for the field and column IDs relevant to your chosen record set.

In [ ]:
# Select a record set and its numeric field

# Use the overview to set these variables appropriately:
RECORD_SET_ID = record_set_ids[0]  # For demonstration; update if desired

# Show available columns for this record set
print(f"Available columns in {RECORD_SET_ID}: {dataframes[RECORD_SET_ID].columns.tolist()}")

# You may need to update these to match the appropriate field @id from your dataset:
NUMERIC_FIELD_ID = dataframes[RECORD_SET_ID].select_dtypes('number').columns[0] if len(dataframes[RECORD_SET_ID].select_dtypes('number').columns) else dataframes[RECORD_SET_ID].columns[0]

threshold = 10
df = dataframes[RECORD_SET_ID]

# Filter if possible (if numeric)
if pd.api.types.is_numeric_dtype(df[NUMERIC_FIELD_ID]):
    filtered = df[df[NUMERIC_FIELD_ID] > threshold]
    print(f"Filtered records with {NUMERIC_FIELD_ID} > {threshold}: {len(filtered)} record(s)")
    display(filtered.head())

    # Normalize field
    filtered[f"{NUMERIC_FIELD_ID}_normalized"] = (
        filtered[NUMERIC_FIELD_ID] - filtered[NUMERIC_FIELD_ID].mean()
    ) / (filtered[NUMERIC_FIELD_ID].std() + 1e-8)
    print(f"Normalized {NUMERIC_FIELD_ID} for filtered records:")
    display(filtered[[NUMERIC_FIELD_ID, f"{NUMERIC_FIELD_ID}_normalized"]].head())

    # Group by a field if a suitable one exists (e.g., first categorical column)
    categorical_fields = df.select_dtypes(include=['object', 'category']).columns
    GROUP_FIELD_ID = None
    if len(categorical_fields):
        GROUP_FIELD_ID = categorical_fields[0]
        print(f"Grouping by field: {GROUP_FIELD_ID}")
        grouped = filtered.groupby(GROUP_FIELD_ID).mean(numeric_only=True)
        print(f"Grouped (mean) data by {GROUP_FIELD_ID}:")
        display(grouped.head())
else:
    print(f"Warning: The field chosen is not numeric. Please review available column names above and select a numeric field.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following is a generic histogram/boxplot for a selected numeric field:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,4))
if pd.api.types.is_numeric_dtype(df[NUMERIC_FIELD_ID]):
    sns.histplot(df[NUMERIC_FIELD_ID], kde=True)
    plt.title(f"Distribution of {NUMERIC_FIELD_ID} in RecordSet {RECORD_SET_ID}")
    plt.xlabel(NUMERIC_FIELD_ID)
    plt.ylabel("Count")
    plt.show()

    # Boxplot if grouped data available
    if 'GROUP_FIELD_ID' in locals() and GROUP_FIELD_ID is not None:
        plt.figure(figsize=(14,4))
        sns.boxplot(x=GROUP_FIELD_ID, y=NUMERIC_FIELD_ID, data=df)
        plt.title(f"{NUMERIC_FIELD_ID} by {GROUP_FIELD_ID}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Selected field is not numeric. Please select a numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to explore a FAIR dataset defined by Croissant metadata. 

You can adapt and expand this notebook to suit the record sets, fields, or columns of interest, always referencing by their Croissant `@id` attributes for reproducible research and data science.


(Notebook complete. Review fields and IDs provided above for your custom analysis.)